# Imports

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime

import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns

from math import sqrt
from sklearn.metrics import mean_squared_error

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX

import sys
import os
sys.path.insert(0, os.path.abspath(".."))

from funcs import test_stationarity, diff

In [ ]:
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [ ]:
mpl.rcParams['axes.labelsize'] = 14
mpl.rcParams['xtick.labelsize'] = 12
mpl.rcParams['ytick.labelsize'] = 12
mpl.rcParams['text.color'] = 'k'
mpl.rcParams['figure.figsize'] = (9,5)
plt.style.use('ggplot')

# Load Data

In [ ]:
df = pd.read_csv("../data/raw/dataset_part2.csv",
                          index_col=1,
                          parse_dates=True,
                          date_format="%d-%m-%Y %H:%M",
                          header=0)

In [ ]:
df.index

In [ ]:
df_train = df[:16057]
df_valid = df[16057:]

# Pre-Processing

In [ ]:
df_train['log'] = np.log(df_train['active_users'])
df_valid['log'] = np.log(df_valid['active_users'])

In [ ]:
df_train.shape, df_valid.shape

In [ ]:
arr_train = df_train['log'].to_numpy()
arr_valid = df_valid['log'].to_numpy()

# Forecasting

In [ ]:
len_forecast = len(df_valid)
len_forecast_future = len(df_train) + len(df_valid)

## ARIMA

In [ ]:
p = 2
d = 1
q = 0

model_arima = ARIMA(endog=df_train['log'], order=(p,d,q), freq='h')

In [ ]:
arima_res = model_arima.fit()

In [ ]:
arima_res.summary()

In [ ]:
arima_res_fv = arima_res.fittedvalues

In [ ]:
plt.plot(df_train['log'])
plt.plot(arima_res_fv)
plt.show()

In [ ]:
arima_forecast = arima_res.forecast(len_forecast)

In [ ]:
plt.plot(df_valid['log'])
plt.plot(arima_forecast)
plt.show()

In [ ]:
max(arima_forecast), min(arima_forecast)

In [ ]:
print(f"RMSE using ARIMA Forecast:\n"
      f"{np.sqrt(sqrt(mean_squared_error(arr_valid, arima_forecast))):.4f}")

## SARIMAX

In [ ]:
p, d, q = 6, 1, 0
seasonal_order = (0,1,1,24)

model_sarimax = SARIMAX(endog=df_train['log'], order=(p,d,q), seasonal_order= seasonal_order, trend='n', freq='h')

In [ ]:
sarimax_res = model_sarimax.fit()

In [ ]:
sarimax_forecast = sarimax_res.forecast(len_forecast)

In [ ]:
max(sarimax_forecast), min(sarimax_forecast)

In [ ]:
plt.plot(df_valid['log'])
plt.plot(sarimax_forecast)
plt.show()

In [ ]:
print(f"RMSE using SARIMAX Forecast:\n"
      f"{np.sqrt(sqrt(mean_squared_error(arr_valid, sarimax_forecast))):.4f}")

# Forecast futuro

In [ ]:
window=23

sarimax_forecast_future = sarimax_res.predict(
    start=len_forecast_future,
    end=len_forecast_future+window)

In [ ]:
sarimax_forecast_future

In [ ]:
# Inverse transform

predicted_values = np.exp(sarimax_forecast_future)
predicted_values

In [ ]:
# Last 4 days of the original dataset
# And the predictions

plt.plot(df_valid['active_users'][-96:])
plt.plot(predicted_values)
plt.show()

# End